In [1]:
# LangChain은 언어 모델(LLM)을 활용해 다양한 어플리케이션을 개발할 수 있는 프레임워크
!pip install -U langchain langchain-core langchain-community langchain-openai
!pip install -U langchain-splitters openai tiktoken
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully u

In [9]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
  raise RuntimeError(".env 파일에 OPENAI_API_KEY가 없어요")

# print("읽기 성공")

In [10]:
from re import search
# LLM 준비
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

#문서 로딩
docs = TextLoader('sample.txt', encoding='utf-8').load()
print(f'문서 갯수:{len(docs)}')
print(docs)
print(docs[0].page_content)

print()
#문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
chunks = splitter.split_documents(docs)
print(f'생성된 청크 수 : {len(chunks)}')
print(chunks)
print(chunks[0].page_content)

문서 갯수:1
[Document(metadata={'source': 'sample.txt'}, page_content='현대자동차그룹이 중국 카이워그룹과 손잡고 중국 광둥성 광저우시에 수소연료전지버스를 공급하며 현지 수소산업 생태계 구축에 앞장선다. 최근 APEC 정상회의, 한중정상회담 등을 통해 한국, 중국 양국 간 우호적 관계가 강화되고 있는 가운데 맺은 경제협력의 결실이다.\n현대차그룹은 중국 수소연료전지시스템법인 ‘HTWO(에이치투) 광저우’가 중국 상용차업체 카이워그룹과 공동 개발한 8.5m 수소연료전지버스가 지난 11일 현지 버스사업 국유기업인 광저우국영버스그룹이 발표한 ‘수소연료전지 도시버스 구매 프로젝트’ 입찰 결과 종합평가 1위로 최종 낙찰에 성공했다고 18일 밝혔다.')]
현대자동차그룹이 중국 카이워그룹과 손잡고 중국 광둥성 광저우시에 수소연료전지버스를 공급하며 현지 수소산업 생태계 구축에 앞장선다. 최근 APEC 정상회의, 한중정상회담 등을 통해 한국, 중국 양국 간 우호적 관계가 강화되고 있는 가운데 맺은 경제협력의 결실이다.
현대차그룹은 중국 수소연료전지시스템법인 ‘HTWO(에이치투) 광저우’가 중국 상용차업체 카이워그룹과 공동 개발한 8.5m 수소연료전지버스가 지난 11일 현지 버스사업 국유기업인 광저우국영버스그룹이 발표한 ‘수소연료전지 도시버스 구매 프로젝트’ 입찰 결과 종합평가 1위로 최종 낙찰에 성공했다고 18일 밝혔다.

생성된 청크 수 : 4
[Document(metadata={'source': 'sample.txt'}, page_content='현대자동차그룹이 중국 카이워그룹과 손잡고 중국 광둥성 광저우시에 수소연료전지버스를 공급하며 현지 수소산업 생태계 구축에 앞장선다. 최근 APEC 정상회의, 한중정상회담 등을 통해'), Document(metadata={'source': 'sample.txt'}, page_content='등을 통해 한국, 중국 양국 간 우호적 관계가 강화되고 있는 가운데 맺은 경제협력의 결실이다.'),

In [15]:
print()
# 임베딩 & 벡터 스토어
# embeddings = OpenAIEmbeddings()   # 유료
# vectorstore = Chroma.from_documents(chunks, embeddings)
# retriever = vectorstore.as_retriever(search_kwargs={'k':3})
# print(retriever)

def format_docs(docs):
  return '\n\n'.join(d.page_content for d in docs)

from langchain_core.runnables import RunnableLambda
docs_runnable = RunnableLambda(lambda _: chunks)  # 고정 청크를 Runnable로 래핑

prompt = ChatPromptTemplate.from_template(
    '답하세요\n'
    '{context}\n'
    '질문:{question}'
)

# 고정된 데이터는 RunnableLambda로 감싸야 한다.
chain =(
    {   # 1) 입력 데이터 구성
        # context 생성 규칙
        # docs_runnable: 실행될 때 항상 'chunks' 리스트를 반환하는 RunnableLambda
        # docs_runnable | RunnableLambda(format_docs)
        #      : chunks 리스트를 문서 문자열(context)로 변환
        'context':docs_runnable | RunnableLambda(format_docs),
        # question 생성 규칙 : 사용자 입력(question)을 그대로 다음 단계로 전달
        'question':RunnablePassthrough()
    }
    # 2) PromptTemplate 적용 : {context, question}을 prompt 템플릿에 채워 넣음
    | prompt
    # 3) 템플릿이 완성되면 llm을 호출해 답변 생성
    | llm
    # 4) LLM 결과에서 message 객체 대신 문자열만 추출하여 텍스트로 출력
    | StrOutputParser()
)

user_question = "현대자동차에 대해 설명해"
answer = chain.invoke(user_question)
print(f'질문 : {user_question}')
print(f'대답 : {answer}')


# 수행 순서 ----------------------------------
# 사용자 질문 ("현대자동차 설명해")
#         │
#         ▼
#  RunnablePassthrough()  → question 유지
#         │
#  docs_runnable → chunks 꺼내기
#         │
#  format_docs → context 문자열 생성
#         │
#         └───> {"context": ..., "question": ...}
#                     │
#                     ▼
#                 PromptTemplate
#                     │
#                     ▼
#                     LLM
#                     │
#                     ▼
#             StrOutputParser()
#                     │
#                     ▼
#                 최종 answer



질문 : 현대자동차에 대해 설명해
대답 : 현대자동차는 한국의 대표적인 자동차 제조업체로, 1967년에 설립되었습니다. 현대자동차는 승용차, 상용차, 스포츠유틸리티차(SUV) 등 다양한 차량을 생산하며, 글로벌 시장에서 활발하게 활동하고 있습니다. 현대차는 기술 혁신과 디자인을 중시하며, 전기차와 수소연료전지차 등 친환경차 개발에도 많은 노력을 기울이고 있습니다.

현대자동차는 전 세계 여러 나라에 생산 시설을 두고 있으며, 다양한 모델을 출시하여 소비자들의 요구를 충족시키고 있습니다. 또한, 현대차는 자율주행 기술, 커넥티드 카 기술 등 미래 자동차 기술 개발에도 적극적으로 투자하고 있습니다.

현대차는 최근 몇 년간 지속 가능한 모빌리티 솔루션을 제공하기 위해 수소연료전지 기술을 강화하고 있으며, 이를 통해 탄소 배출을 줄이고 친환경 교통 수단을 발전시키는 데 기여하고자 하고 있습니다. 이러한 노력은 글로벌 자동차 산업의 변화에 발맞추어 나가기 위한 전략의 일환입니다.
